# Jacobian determinant in the density push-forward formula

This notebook generates `fig:monge-jacobian-pushforward-density`.  It illustrates the change-of-variables formula

$$
\rho_\beta(T(x)) = \frac{\rho_\alpha(x)}{|\det DT(x)|},
$$

for a smooth diffeomorphism of the unit square.  The source density is uniform, so any nonuniformity in the target density is entirely caused by the Jacobian determinant.  The map bends the square grid and compresses a narrow strip; this compressed region becomes visibly denser in the target panel.


In [1]:
from pathlib import Path
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, to_rgb
from PIL import Image
import subprocess
import tempfile

ROOT = Path.cwd()
if not (ROOT / "notebooks-figures").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "notebooks-figures"))

from figure_style import BLUE, RED, VIOLET, interp_color, figure_dir, remove_axes, save_pdf, setup_matplotlib

setup_matplotlib()

NAME = "monge-jacobian-pushforward-density"
OUT = figure_dir(NAME)
ARXIV_OUT = ROOT / "arxiv" / "figures"
THUMB_OUT = ROOT / "notebooks-figures" / "thumbnails"
ARXIV_OUT.mkdir(parents=True, exist_ok=True)
THUMB_OUT.mkdir(parents=True, exist_ok=True)


## A simple diffeomorphism with analytic determinant

We use a triangular diffeomorphism.  Its first coordinate compresses near the middle of the square,

$$\nu(x)=x+\frac{a}{2\pi}\sin(2\pi x),\qquad \nu'(x)=1+a\cos(2\pi x)>0,$$

and the second coordinate bends horizontal grid lines while keeping the boundary of the square fixed,

$$T(x,y)=\bigl(\nu(x),\; y+b\sin(2\pi \nu(x))\sin^2(\pi y)\bigr).$$

The determinant is

$$
\det DT(x,y)=\nu'(x)\left(1+b\pi\sin(2\pi\nu(x))\sin(2\pi y)\right),
$$

which stays positive for the chosen values of `a` and `b`.


In [2]:
a = 0.70
b = 0.14


def nu(x):
    return x + a * np.sin(2 * np.pi * x) / (2 * np.pi)


def nup(x):
    return 1 + a * np.cos(2 * np.pi * x)


def Txy(x, y):
    s = nu(x)
    z = y + b * np.sin(2 * np.pi * s) * np.sin(np.pi * y) ** 2
    return s, z


def det_jacobian(x, y):
    s = nu(x)
    return nup(x) * (1 + b * np.pi * np.sin(2 * np.pi * s) * np.sin(2 * np.pi * y))

# Sanity check on a fine grid: the map is orientation preserving.
grid = np.linspace(0, 1, 501)
X, Y = np.meshgrid(grid, grid, indexing="xy")
J = det_jacobian(X, Y)
print("determinant range", float(J.min()), float(J.max()))
assert J.min() > 0


determinant range 0.2988790116301329 2.160394788614578


## Inverting the map on a grid

Since the first coordinate only depends on `x`, we invert it by interpolation.  For each target abscissa, the second coordinate is a monotone scalar map in `y`, so a short vectorized bisection computes the inverse.  The target density is then evaluated from the determinant formula.


In [3]:
def inverse_map(Xt, Yt):
    # Vectorized inverse of T on the unit square.
    x_grid = np.linspace(0, 1, 5001)
    u_grid = nu(x_grid)
    x = np.interp(np.ravel(Xt), u_grid, x_grid).reshape(np.shape(Xt))
    s = nu(x)
    coeff = b * np.sin(2 * np.pi * s)
    lo = np.zeros_like(Yt)
    hi = np.ones_like(Yt)
    for _ in range(48):
        mid = 0.5 * (lo + hi)
        val = mid + coeff * np.sin(np.pi * mid) ** 2
        lo = np.where(val < Yt, mid, lo)
        hi = np.where(val >= Yt, mid, hi)
    y = 0.5 * (lo + hi)
    return x, y

N = 360
u_grid = np.linspace(0, 1, N)
Xt, Yt = np.meshgrid(u_grid, u_grid, indexing="xy")
Xi, Yi = inverse_map(Xt, Yt)
rho_beta = 1.0 / det_jacobian(Xi, Yi)
print("target density range", float(rho_beta.min()), float(rho_beta.max()), "mass", float(rho_beta.mean()))


target density range 0.46287629784209566 3.342895823179868 mass 0.9988562592782053


## Draw the source grid and the pushed-forward density

The source panel shows the unit square with a colored grid.  The target panel overlays the deformed image of that grid with the density image and its level sets.  The blue high-density band is located where the diffeomorphism compresses area.


In [4]:
import matplotlib.patheffects as pe


def blend(color, amount=0.55):
    rgb = np.array(to_rgb(color))
    return tuple((1 - amount) * rgb + amount * np.ones(3))


def draw_square_frame(ax, lw=0.80):
    ax.plot([0, 1, 1, 0, 0], [0, 0, 1, 1, 0], color="#242424", lw=lw, zorder=10)


def grid_color(t):
    if t <= 0.5:
        return interp_color(t / 0.5, RED, VIOLET)
    return interp_color((t - 0.5) / 0.5, VIOLET, BLUE)


def draw_grid(ax, transform=None, n=14, samples=360, alpha=0.88, lw=0.60):
    vals = np.linspace(0, 1, n)
    s = np.linspace(0, 1, samples)
    for v in vals:
        col = grid_color(v)
        x = np.full_like(s, v)
        y = s.copy()
        if transform is not None:
            x, y = transform(x, y)
        ax.plot(x, y, color=col, lw=lw, alpha=alpha, zorder=6)
    for v in vals:
        col = blend(grid_color(v), 0.42)
        x = s.copy()
        y = np.full_like(s, v)
        if transform is not None:
            x, y = transform(x, y)
        ax.plot(x, y, color=col, lw=0.50 * lw / 0.60, alpha=0.72 * alpha, zorder=6)


def draw_source_panel(path):
    fig, ax = plt.subplots(figsize=(2.55, 2.55))
    remove_axes(ax)
    ax.set_aspect("equal")
    ax.set_xlim(-0.018, 1.018)
    ax.set_ylim(-0.018, 1.018)
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, facecolor=blend(RED, 0.90), edgecolor="none", zorder=0))
    draw_grid(ax, transform=None, n=14, samples=220, alpha=0.92, lw=0.62)
    draw_square_frame(ax)
    save_pdf(fig, path, pad_inches=0.016)
    plt.close(fig)


def draw_target_panel(path):
    fig, ax = plt.subplots(figsize=(2.55, 2.55))
    remove_axes(ax)
    ax.set_aspect("equal")
    ax.set_xlim(-0.018, 1.018)
    ax.set_ylim(-0.018, 1.018)

    cmap = LinearSegmentedColormap.from_list(
        "density_blue",
        ["#ffffff", "#eef4fb", "#bdd2ea", "#729bc8", BLUE],
    )
    vmax = np.quantile(rho_beta, 0.975)
    ax.imshow(
        np.clip(rho_beta, 0, vmax),
        extent=[0, 1, 0, 1],
        origin="lower",
        cmap=cmap,
        vmin=0,
        vmax=vmax,
        interpolation="bicubic",
        zorder=0,
    )
    draw_grid(ax, transform=Txy, n=14, samples=520, alpha=0.66, lw=0.52)
    levels = np.quantile(rho_beta, np.linspace(0.42, 0.94, 7))
    contours = ax.contour(
        Xt,
        Yt,
        rho_beta,
        levels=np.unique(levels),
        colors="#1f2d3a",
        linewidths=0.50,
        alpha=0.70,
        zorder=8,
    )
    contours.set_path_effects([pe.Stroke(linewidth=0.86, foreground="white", alpha=0.30), pe.Normal()])
    draw_square_frame(ax)
    save_pdf(fig, path, pad_inches=0.016)
    plt.close(fig)


def copy_to_arxiv(pdf_name):
    shutil.copyfile(OUT / pdf_name, ARXIV_OUT / f"{NAME}--{pdf_name}")


draw_source_panel(OUT / "source-grid.pdf")
draw_target_panel(OUT / "target-density-grid.pdf")
copy_to_arxiv("source-grid.pdf")
copy_to_arxiv("target-density-grid.pdf")


In [5]:
# Thumbnail for the notebook gallery.
def render_pdf_to_image(pdf, scale=3.2):
    try:
        import fitz
        doc = fitz.open(pdf)
        page = doc[0]
        pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
        im = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        doc.close()
        return im
    except Exception:
        with tempfile.TemporaryDirectory() as tmp:
            prefix = Path(tmp) / "panel"
            subprocess.run(
                ["pdftoppm", "-png", "-r", str(int(72 * scale)), str(pdf), str(prefix)],
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            return Image.open(f"{prefix}-1.png").convert("RGB")

imgs = [render_pdf_to_image(OUT / "source-grid.pdf"), render_pdf_to_image(OUT / "target-density-grid.pdf")]
h = max(im.height for im in imgs)
pad = 14
canvas = Image.new("RGB", (sum(im.width for im in imgs) + 3 * pad, h + 2 * pad), "white")
x0 = pad
for im in imgs:
    canvas.paste(im, (x0, pad + (h - im.height) // 2))
    x0 += im.width + pad
thumb = THUMB_OUT / f"{NAME}.png"
canvas.save(thumb, quality=94)
print(thumb)


/Users/gpeyre/Dropbox/github/ot4ml/notebooks-figures/thumbnails/monge-jacobian-pushforward-density.png
